# Create TA ČR Awards (Technology Agency of the Czech Republic)

Creates awards for the **Technology Agency of the Czech Republic (TA ČR)** from TA ČR's own R&D
search engine **STARFOS** (https://starfos.tacr.cz), which indexes every Czech publicly funded
project in the national CEP register (IS VaVaI). The script uses STARFOS's own JSON backend
(old.starfos.tacr.cz/api/starfos, robots "Allow: /"): the site's **Export** endpoint (CSV, one
export per programme, each checked against the search API's `num_found`) plus the search API for
exact implementation dates.

**Shared portal, routed by provider (runbook §2.3.2).** STARFOS covers ~62,700 projects from all
Czech providers (GAČR, ministries, regions ...). Only the 24 programmes STARFOS files under the
provider **"TA ČR - Technology Agency of the Czech Republic"** are ingested (ALFA TA, BETA TB/TI/TT,
OMEGA TD, Centra kompetence TE, DELTA TF/TM, GAMA TG/TP, EPSILON TH, ZETA TJ, THETA TK/TS, ETA TL,
NCK TN, KAPPA TO, SIGMA TQ, TREND FW, DOPRAVA CK/CL, Prostředí pro život SS/SQ, PRODEF OZ) --
5,608 projects 2011-2026. TA ČR is the formal provider of all of them, including the programmes it
administers for ministries (TREND/MPO, DOPRAVA/MD, Prostředí pro život/MŽP); these codes are
cited to TA ČR. No existing direct ingest carries these codes (MPO/MZe/MV/MŠMT/GAČR ingests use
other programme prefixes). All programmes are applied research -- no filter.

**Fields:** `funder_award_id` = CEP project code (e.g. `TK01010001`, `FW03010134`; BETA3 public
contracts look like `TTAERU506`) -- the code papers cite (939 of 1,454 distinct F4320315323
citation-stub ids collapse). `amount` = "Public support" (state support from TA ČR over the project
lifetime), published in thousand CZK, converted to **CZK**. Title/description English, Czech
fallback. PI = first "main project researcher" (Czech academic titles stripped in the script;
ORCID when STARFOS has it); affiliation = main participant. `investigators` = all main + other
researchers (lead first). `funder_scheme` = programme.

**Twin funder row:** F4320321623 "Technologická Agentura České Republiky" (ror 04v0fk911,
doi 10.13039/501100002969) carries 890 distinct stub ids, 575 of which are codes in this ingest;
they will NOT collapse (award ids hash the funder id). This ingest ships under the tracker's
F4320315323 (doi 10.13039/100014809, no ROR) -- coordinator to decide on a funder merge.

**Prerequisites:** run `scripts/local/tacr_starfos_to_s3.py` (uploads
`s3://openalex-ingest/awards/tacr/tacr_projects.parquet`, §1.4 shrink guard).

**Funder (Path A, F4320*):** funder_id `4320315323` (Technology Agency of the Czech Republic),
read from `openalex.funders.funders`.

**Priority:** `534` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tacr_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/tacr/tacr_projects.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects, COUNT(DISTINCT project_code) as distinct_codes
FROM openalex.awards.tacr_raw;

In [ ]:
%sql
DESCRIBE openalex.awards.tacr_raw;

In [ ]:
%sql
SELECT programme_code, programme, provider, COUNT(*) AS n,
       ROUND(SUM(TRY_CAST(public_support_czk AS DOUBLE)), 0) AS support_czk
FROM openalex.awards.tacr_raw
GROUP BY 1, 2, 3 ORDER BY n DESC;

## Step 1.6: Funder existence check (Path A)
F4320315323 must resolve to exactly one row in `openalex.funders.funders`; if not, STOP.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320315323;

## Step 2: Create TA ČR Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.tacr_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320315323  -- Technology Agency of the Czech Republic
),
g AS (
    SELECT
        r.*,
        TRY_TO_DATE(r.start_date, 'yyyy-MM-dd') AS start_d,
        TRY_TO_DATE(r.end_date, 'yyyy-MM-dd') AS end_d,
        COALESCE(from_json(r.investigators_json,
            'array<struct<name_raw:string,given_name:string,family_name:string,orcid:string,vedidk:string>>'),
            array()) AS people
    FROM openalex.awards.tacr_raw r
    WHERE r.project_code IS NOT NULL AND TRIM(r.project_code) != ''
),
p AS (
    -- struct field order MUST match openalex_awards_raw: given, family, orcid, role_start, affiliation
    SELECT
        g.*,
        CASE WHEN g.lead_family_name IS NOT NULL THEN named_struct(
            'given_name', NULLIF(TRIM(g.lead_given_name), ''),
            'family_name', NULLIF(TRIM(g.lead_family_name), ''),
            'orcid', CASE WHEN g.lead_orcid RLIKE '^[0-9]{4}-' THEN CONCAT('https://orcid.org/', g.lead_orcid) END,
            'role_start', CAST(NULL AS DATE),
            'affiliation', named_struct(
                'name', NULLIF(TRIM(g.lead_institution), ''),
                'country', NULLIF(TRIM(g.lead_country), ''),
                'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
            )
        ) END AS lead,
        TRANSFORM(g.people, x -> named_struct(
            'given_name', NULLIF(TRIM(x.given_name), ''),
            'family_name', NULLIF(TRIM(x.family_name), ''),
            'orcid', CASE WHEN x.orcid RLIKE '^[0-9]{4}-' THEN CONCAT('https://orcid.org/', x.orcid) END,
            'role_start', CAST(NULL AS DATE),
            'affiliation', CAST(NULL AS STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>)
        )) AS team
    FROM g
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(p.project_code))))) % 9000000000 as id,
    COALESCE(NULLIF(TRIM(p.title_en), ''), NULLIF(TRIM(p.title_cs), '')) as display_name,
    COALESCE(NULLIF(TRIM(p.description_en), ''), NULLIF(TRIM(p.description_cs), '')) as description,
    f.funder_id,
    TRIM(p.project_code) as funder_award_id,
    TRY_CAST(p.public_support_czk AS DOUBLE) as amount,
    'CZK' as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    NULLIF(TRIM(p.programme), '') as funder_scheme,
    'starfos_tacr' as provenance,
    p.start_d as start_date,
    p.end_d as end_date,
    COALESCE(YEAR(p.start_d), TRY_CAST(p.start_year AS INT)) as start_year,
    COALESCE(YEAR(p.end_d), TRY_CAST(p.end_year AS INT)) as end_year,
    p.lead as lead_investigator,
    CASE WHEN TRY_CAST(p.n_main_researchers AS INT) >= 2 THEN try_element_at(p.team, 2) END as co_lead_investigator,
    CASE WHEN p.lead IS NOT NULL AND size(p.team) > 0
         THEN concat(array(p.lead), slice(p.team, 2, size(p.team) - 1))
         WHEN size(p.team) > 0 THEN p.team END as investigators,
    p.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(p.project_code))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM p
CROSS JOIN src_funder f;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'starfos_tacr' AND priority = 534;

-- Insert into openalex_awards_raw with priority 534 (direct funder ingest; higher wins
-- under the 2026-06-20 DESC dedup, so it outranks the priority-0 acknowledgement shells).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    534 as priority
FROM openalex.awards.tacr_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.tacr_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_date, end_date,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.orcid,
       lead_investigator.affiliation.name, size(investigators) AS team
FROM openalex.awards.tacr_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_czk
FROM openalex.awards.tacr_awards
GROUP BY funder_scheme ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_czk
FROM openalex.awards.tacr_awards
WHERE start_year IS NOT NULL GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency checks: PI and title top-20 must be a real long tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.tacr_awards
WHERE lead_investigator IS NOT NULL
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.tacr_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.7 coverage.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_date) as has_start_date,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator) as has_pi,
    COUNT(lead_investigator.orcid) as has_pi_orcid,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.tacr_awards;

In [ ]:
%sql
-- Shape check (2.1.1): CEP project codes only -- two-letter programme prefix + digits
-- (BETA3 public contracts TTxxxx are alphanumeric).
SELECT
    SUM(CASE WHEN funder_award_id RLIKE '^[A-Z]{2}[0-9]{6,8}$' THEN 1 ELSE 0 END) AS cep_codes,
    SUM(CASE WHEN funder_award_id RLIKE '^TT[A-Z]+[0-9]+$' THEN 1 ELSE 0 END) AS beta3_codes,
    SUM(CASE WHEN NOT (funder_award_id RLIKE '^[A-Z]{2}[0-9]{6,8}$' OR funder_award_id RLIKE '^TT[A-Z]+[0-9]+$') THEN 1 ELSE 0 END) AS other,
    COUNT(*) AS total
FROM openalex.awards.tacr_awards;

In [ ]:
%sql
-- Citation stubs (priority < 3) under F4320315323 that collapse onto this ingest, and the
-- twin F4320321623 stubs whose codes are in this ingest (these do NOT collapse).
SELECT c.funder_id, c.provenance, COUNT(*) as stub_rows,
       COUNT(t.id) as collapse_onto_tacr,
       COUNT(t2.funder_award_id) as code_in_tacr_ingest
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.tacr_awards t ON t.id = c.id
LEFT JOIN (SELECT DISTINCT LOWER(funder_award_id) AS funder_award_id FROM openalex.awards.tacr_awards) t2
       ON t2.funder_award_id = LOWER(TRIM(c.funder_award_id))
WHERE c.funder_id IN (4320315323, 4320321623) AND c.priority < 3
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'starfos_tacr'
GROUP BY provenance, priority;